In [1]:
import os

In [2]:
%pwd

'/home/user/Documents/Kidney_Disease Using Deep Learning/research'

In [3]:
os.chdir("../")

In [4]:
%pwd

'/home/user/Documents/Kidney_Disease Using Deep Learning'

In [5]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class PrepareBaseModelConfig:
    root_dir: Path
    base_model_path: Path
    updated_base_model_path: Path
    params_image_size: int
    params_learning_rate: float
    params_include_top: bool
    params_weights: str
    params_classes: int    

In [6]:
from src.cnnClassifier.constants import *
from src.cnnClassifier.utils.common import read_yaml, create_directories

ModuleNotFoundError: No module named 'box'

In [ ]:
class ConfigurationManager:
    def __init__(
        self,
        config_filepath=CONFIG_FILE_PATH,
        params_filepath=PARAMS_FILE_PATH
    ):
        
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)

        create_directories([self.config.artifacts_root])

    def get_prepare_base_model_config(self) -> PrepareBaseModelConfig:
        config = self.config.prepare_base_model 
        create_directories([config.root_dir])

        prepare_base_model_config = PrepareBaseModelConfig(
            root_dir=Path(config.root_dir),
            base_model_path=Path(config.base_model_path),
            updated_base_model_path=Path(config.updated_base_model_path),
            params_image_size=self.params.IMAGE_SIZE,
            params_learning_rate=self.params.LEARNING_RATE,
            params_include_top=self.params.INCLUDE_TOP,
            params_weights=self.params.WEIGHTS,
            params_classes=self.params.CLASSES
        )
      
        return prepare_base_model_config

In [ ]:
import os
import urllib.request as request
from zipfile import ZipFile
import tensorflow as tf

In [ ]:
class PrepareBaseModel:
    def __init__(self, config: PrepareBaseModelConfig):
        self.config = config

    
    def get_base_model(self):
        # ensure input_shape is a tuple (H, W, C)
        input_shape = (self.config.params_image_size, self.config.params_image_size, 3) if isinstance(self.config.params_image_size, int) else tuple(self.config.params_image_size)
        try:
            from tensorflow import keras as _keras
        except Exception:
            import keras as _keras

        self.model = _keras.applications.vgg16.VGG16(
            input_shape=input_shape,
            weights=self.config.params_weights,
            include_top=self.config.params_include_top
        )

        self.save_model(path=self.config.base_model_path, model=self.model)

    

    @staticmethod
    def _prepare_full_model(model, classes, freeze_all, freeze_till, learning_rate):
        try:
            from tensorflow import keras as _keras
        except Exception:
            import keras as _keras

        if freeze_all:
            for layer in model.layers:
                layer.trainable = False
        elif (freeze_till is not None) and (freeze_till > 0):
            for layer in model.layers[:-freeze_till]:
                layer.trainable = False

        flatten_in = _keras.layers.Flatten()(model.output)
        prediction = _keras.layers.Dense(
            units=classes,
            activation="softmax"
        )(flatten_in)

        full_model = _keras.models.Model(
            inputs=model.input,
            outputs=prediction
        )

        full_model.compile(
            optimizer=_keras.optimizers.SGD(learning_rate=learning_rate),
            loss=_keras.losses.CategoricalCrossentropy(),
            metrics=["accuracy"]
        )

        full_model.summary()
        return full_model
    
    
    def update_base_model(self):
        self.full_model = self._prepare_full_model(
            model=self.model,
            classes=self.config.params_classes,
            freeze_all=True,
            freeze_till=None,
            learning_rate=self.config.params_learning_rate
        )

        self.save_model(path=self.config.updated_base_model_path, model=self.full_model)

    @staticmethod
    def save_model(path: Path, model):
        model.save(path)

In [ ]:
try:
    config = ConfigurationManager()
    prepare_base_model_config = config.get_prepare_base_model_config()
    prepare_base_model = PrepareBaseModel(config=prepare_base_model_config)
    prepare_base_model.get_base_model()
    prepare_base_model.update_base_model()
except Exception as e:
    raise e

2026-04-22 14:06:00,155 - INFO - cnnClassifier
2026-04-22 14:06:00,156 - INFO - cnnClassifier
2026-04-22 14:06:00,158 - INFO - cnnClassifier
2026-04-22 14:06:00,160 - INFO - cnnClassifier


ModuleNotFoundError: No module named 'tensorflow.python'

In [ ]:
import os
print(os.getcwd())

In [ ]:
from pathlib import Path

print(Path("config/config.yaml").exists())
print(Path("../config/config.yaml").exists())
print(Path("../../config/config.yaml").exists())

In [ ]:
Path("../../config/config.yaml").exists()